# Kochi-LLM v1.0 — publish checkpoints to Hugging Face

Converts the **most-trained** checkpoint of every attached Kaggle run to bf16 safetensors and pushes it
to `<your-hf-user>/KochiLLM-v1.0` as its own git revision, tagged `run-NN`. `main` advances only when a
run has a **higher step count** than what is already published, so this notebook is **order-independent**
and safe to re-run.

**Scope: weights only.** The model card, `ARCHITECTURE.md`, `TRAINING.md`, `SAMPLES.md`, the loss-curve
assets, `LICENSE`, and the source files are already published in the repo. This notebook will **not**
overwrite an existing `README.md`. It adds `model.safetensors`, `config.json`, and `metadata.json`
per revision.

## Why you may need several sessions

Each checkpoint is 4.1 GB, so attaching all nine at once (~37 GB) will exceed Kaggle's notebook input
limit. Attach as many as the limit allows, run this notebook, then attach the next batch and run it
again. Already-published revisions are simply overwritten with identical content.

## Setup

- Attach one or more Kaggle notebook outputs containing `best_model.pth` / `latest_backup_model.pth`
- **Internet ON**. No GPU needed — all work is CPU + network
- Add an `HF_TOKEN` Kaggle secret with **write** access


In [ ]:
%%bash
set -e
pip install -q -U huggingface_hub safetensors

DIRS=$(find /kaggle/input \( -name 'best_model.pth' -o -name 'latest_backup_model.pth' \) | xargs -r -n1 dirname | sort -u)
echo "$DIRS"
N=$(printf '%s' "$DIRS" | grep -c . || true)
echo "found $N run dir(s)"
[ "$N" -eq 0 ] && { echo 'NO RUNS ATTACHED'; ls -R /kaggle/input | head -40; exit 1; }

PY_SRC=''
for d in $DIRS /kaggle/input/datasets/dadhichisarker/kochi-llm/model; do
  if [ -f "$d/architecture.py" ]; then PY_SRC=$d; break; fi
done
[ -z "$PY_SRC" ] && { echo 'NO PY SOURCE'; exit 1; }
cp "$PY_SRC"/*.py /kaggle/working/
find /kaggle/input -name 'custom_bpe_tokenizer.pkl' -exec cp {} /kaggle/working/ \; 2>/dev/null || true
ls -lh /kaggle/working/*.py /kaggle/working/*.pkl 2>/dev/null || true


## Preflight

Guards against `huggingface_hub` API drift. In 0.36 several methods made their trailing parameters
keyword-only, which turns a positional call into a runtime `TypeError` — previously this would have
failed only *after* multi-GB uploads were staged. Verify the interpreter before doing any work.


In [ ]:
import inspect
from huggingface_hub import HfApi, CommitOperationAdd

REQUIRED = {
    'upload_file':      ['path_or_fileobj', 'path_in_repo', 'repo_id', 'commit_message'],
    'create_commit':    ['repo_id', 'operations', 'commit_message', 'revision'],
    'create_tag':       ['repo_id', 'tag', 'revision'],
    'create_branch':    ['repo_id', 'branch', 'exist_ok'],
    'create_repo':      ['repo_id', 'repo_type', 'exist_ok'],
    'whoami':           [],
    'list_repo_refs':   ['repo_id'],
    'repo_info':        ['repo_id'],
}

import huggingface_hub
print('huggingface_hub', huggingface_hub.__version__)

bad = []
for meth, params in REQUIRED.items():
    sig = inspect.signature(getattr(HfApi, meth))
    names = set(sig.parameters)
    missing = [p for p in params if p not in names]
    if missing:
        bad.append('%s() missing %s' % (meth, missing))
    for p in params:
        if p in names and sig.parameters[p].kind is inspect.Parameter.POSITIONAL_ONLY:
            bad.append('%s(): %s is positional-only' % (meth, p))

ops_sig = set(inspect.signature(CommitOperationAdd.__init__).parameters)
for p in ('path_in_repo', 'path_or_fileobj'):
    if p not in ops_sig:
        bad.append('CommitOperationAdd missing %s' % p)

if bad:
    raise SystemExit('huggingface_hub API drift:\n  ' + '\n  '.join(bad))
print('preflight OK - all hub calls in this notebook bind')


## Upload plan

Check this table before anything ships.

`chosen` is the checkpoint with the **highest step count**, not necessarily `best_model.pth`: several
early runs carry a stale Phase-1 `best_model.pth` while their real weights sit in
`latest_backup_model.pth`.


In [ ]:
import glob, os, re
import torch

WEIGHTS = ('best_model.pth', 'latest_backup_model.pth')

dirs = sorted({os.path.dirname(p)
               for p in glob.glob('/kaggle/input/**/*.pth', recursive=True)
               if os.path.basename(p) in WEIGHTS})

used = set()


def tag_for(slug, idx):
    nums = re.findall(r'\d+', slug)
    base = 'run-%02d' % (int(nums[-1]) if nums else idx)
    tag, n = base, 2
    while tag in used:                       # guard: slugs can share a trailing number
        tag = '%s-%d' % (base, n)
        n += 1
    used.add(tag)
    return tag


plan = []
for i, d in enumerate(dirs, 1):
    cands = []
    for n in WEIGHTS:
        fp = os.path.join(d, n)
        if os.path.exists(fp):
            ck = torch.load(fp, map_location='cpu', weights_only=False, mmap=True)
            cands.append((int(ck.get('iter', -1)), fp, ck))
    if not cands:
        continue
    it, fp, ck = max(cands, key=lambda t: t[0])
    slug = os.path.basename(d.rstrip('/'))
    plan.append({'dir': d, 'slug': slug, 'branch': tag_for(slug, i),
                 'iter': it, 'path': fp, 'ckpt': ck})

print('%-36s %-10s %7s %8s  %s' % ('source run', 'branch', 'iter', 'val', 'chosen file'))
for e in sorted(plan, key=lambda e: e['iter']):
    v = e['ckpt'].get('val_loss')
    v = float(v) if v is not None else float('nan')
    print('%-36s %-10s %7d %8.4f  %s' % (e['slug'][:36], e['branch'], e['iter'], v,
                                        os.path.basename(e['path'])))
print('\n%d run(s) will be pushed.' % len(plan))


In [ ]:
import os
from huggingface_hub import HfApi

TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as e:
    print('no Kaggle secret:', e)
if not TOKEN:
    TOKEN = os.environ.get('HF_TOKEN')
if not TOKEN:
    raise SystemExit('Add an HF_TOKEN Kaggle secret (write access) and re-run.')

api = HfApi(token=TOKEN)
USER = api.whoami()['name']
REPO = USER + '/KochiLLM-v1.0'
api.create_repo(REPO, repo_type='model', exist_ok=True)
print('target repo:', REPO)

CODE_FILES = ['architecture.py', 'tokenizer.py', 'data_loader.py',
              'generate.py', 'prepare_data.py', 'train_custom.py',
              'custom_bpe_tokenizer.pkl']

existing = {s.rfilename for s in (api.repo_info(REPO).siblings or [])}
print('repo already has %d file(s)' % len(existing))

added = [f for f in CODE_FILES if os.path.exists(f) and f not in existing]
for f in added:
    api.upload_file(path_or_fileobj=f, path_in_repo=f, repo_id=REPO,
                    commit_message='Add ' + f)
print('uploaded code:', added or 'nothing (already present)')

if 'README.md' in existing:
    print('README.md already published - leaving it untouched.')
else:
    import json as _json
    with open('README.md', 'w', encoding='utf-8') as f:
        f.write(_json.loads("---\nlibrary_name: pytorch\ntags:\n  - moe\n  - mixture-of-experts\n  - gqa\n  - multi-token-prediction\n  - from-scratch\n  - custom-architecture\nlanguage:\n  - en\nlicense: mit\n---\n\n# KochiLLM v1.0\n\nA **1.02B-parameter Mixture-of-Experts language model trained entirely from scratch** on 2x Tesla T4\n(~52 GPU-hours), including its own BPE tokenizer, data pipeline, and distributed training setup.\n\nFinal validation loss **3.96** after exactly one epoch over 180.2M tokens of FineWeb-Edu +\nCosmopedia-v2 (0.176 tokens/param). Weights are bf16.\n\n## Loading\n\nNot an `AutoModel` release - the architecture is custom code and the tokenizer is a custom pickle.\n\n```python\nimport json, torch\nfrom safetensors.torch import load_file\nfrom architecture import MyCustomLLM\nfrom tokenizer import MyCustomTokenizer\n\nconfig = json.load(open('config.json'))\ntok = MyCustomTokenizer(vocab_size=config['vocab_size'])\ntok.load('custom_bpe_tokenizer.pkl')\n\nmodel = MyCustomLLM(**config)                 # re-ties embed.weight <-> output.weight\nsd = load_file('model.safetensors')\nsd['output.weight'] = sd['embed.weight']      # dropped at save time\nmodel.load_state_dict(sd, strict=False)\nmodel.eval().cuda().to(torch.bfloat16)\n```\n\n## Limitations\n\n**No factual knowledge.** At 0.176 tokens/param the model learned fluency and style, not facts;\ngenerated names and dates are plausible and wrong. **Not instruction-tuned** - it continues documents\nrather than answering questions. Context is 1024 tokens. Validation was in-distribution. Full\narchitecture and training documentation is in `ARCHITECTURE.md` and `TRAINING.md`.\n\n## Source\n\n<https://github.com/Dadhichi-Sarker-Shayon/KochiLLM-v1.0>\n\nMIT licensed.\n"))
    api.upload_file(path_or_fileobj='README.md', path_in_repo='README.md',
                    repo_id=REPO, commit_message='Add fallback model card')
    print('wrote fallback model card (no README existed)')


## Notes

- **Tied weights.** `embed.weight` and `output.weight` share storage, which `safetensors` refuses to
  serialise twice. `output.weight` is dropped and recorded in `metadata.json`. Rebuild it with
  `model.embed.weight = model.output.weight` — already done inside `MyCustomLLM.__init__`.
- **bf16.** Cast down from the fp32 checkpoint, halving size to ~2.05 GB. Intended for inference, not
  for resuming training.
- **`config.json` stays pure.** It is written verbatim from the checkpoint so that
  `MyCustomLLM(**config)` keeps working. Do not add `model_type` / `architectures` keys to it.
- **Not an `AutoModel` release.** Custom architecture and a custom pickle tokenizer; see the model
  card for loading instructions.


In [ ]:
import gc, json, os
import torch
from safetensors.torch import save_file
from huggingface_hub import CommitOperationAdd, hf_hub_download


def to_bf16(v):
    # explicit new allocation: .to(bfloat16) is a no-op when v is already bf16,
    # which would leave the result aliasing the mmap'd checkpoint storage
    out = torch.empty(v.shape, dtype=torch.bfloat16)
    out.copy_(v)
    return out


def build_ops(ckpt, entry):
    sd = ckpt['model_state_dict']
    seen, kept, dropped = set(), {}, []
    for k, v in sd.items():
        if v.data_ptr() in seen:
            dropped.append(k)
            continue
        seen.add(v.data_ptr())
        kept[k] = to_bf16(v)

    save_file(kept, 'model.safetensors', metadata={'format': 'pt'})

    with open('config.json', 'w', encoding='utf-8') as f:
        json.dump(ckpt.get('config', {}), f, indent=2)

    meta = {
        'iter': int(ckpt.get('iter', -1)),
        'val_loss': float(ckpt.get('val_loss', float('nan'))),
        'latest_val_loss': float(ckpt.get('latest_val_loss', float('nan'))),
        'source_run': entry['slug'],
        'dtype': 'bfloat16',
        'tied_weights_dropped': dropped,
        'note': 'embed.weight is tied to output.weight; dropped to satisfy safetensors',
    }
    with open('metadata.json', 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=2)

    return [
        CommitOperationAdd(path_in_repo='model.safetensors',
                           path_or_fileobj='model.safetensors'),
        CommitOperationAdd(path_in_repo='config.json',
                           path_or_fileobj='config.json'),
        CommitOperationAdd(path_in_repo='metadata.json',
                           path_or_fileobj='metadata.json'),
    ]


main_iter = -1
try:
    main_iter = json.load(open(hf_hub_download(REPO, 'metadata.json',
                                              revision='main'))).get('iter', -1)
except Exception:
    pass
print('main is currently at iter', main_iter)
print('pushing', len(plan), 'run(s) to', REPO)

for e in sorted(plan, key=lambda e: e['iter']):
    branch = e['branch']
    ops = build_ops(e['ckpt'], e)
    val = float(e['ckpt'].get('val_loss', float('nan')))
    msg = '%s: iter %d, val %.4f (from %s)' % (branch, e['iter'], val, e['slug'])

    try:
        api.create_branch(REPO, branch=branch, exist_ok=True)
    except Exception as ex:
        print('  branch create skipped:', ex)

    api.create_commit(REPO, operations=ops, commit_message=msg, revision=branch)
    api.create_tag(REPO, tag=branch, revision=branch)
    print('pushed', branch, '| iter', e['iter'], '| val %.4f' % val)

    if e['iter'] >= main_iter:
        api.create_commit(REPO, operations=ops, commit_message=msg + ' [main]',
                          revision='main')
        print('  main updated to iter', e['iter'])
        main_iter = e['iter']

    for f in ('model.safetensors', 'config.json', 'metadata.json'):
        if os.path.exists(f):
            os.remove(f)
    del ops
    gc.collect()

print('done')


In [ ]:
refs = api.list_repo_refs(REPO)
print('repo:', REPO)
print('branches:', [b.name for b in refs.branches])
print('tags:     ', [t.name for t in refs.tags])
print('\nfiles on main:')
for s in sorted(api.repo_info(REPO).siblings, key=lambda x: x.rfilename):
    print('  ', s.rfilename)
